In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
pd.set_option('display.max_columns',None)

In [ ]:
parking_area = pd.read_csv('수원도시공사_공영주차장 현황_20241231.csv', encoding='cp949')

In [ ]:
parking_area_addr=parking_area[['주차장명','주차장구분','주차장유형','소재지도로명주소','소재지지번주소','주차구획수','운영요일']]

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
import json
import requests
import os
from dotenv import load_dotenv

load_dotenv()
KAKAO_API_KEY = os.getenv("KAKAO_API_KEY")
api_key = KAKAO_API_KEY


def get_lat_lng(addr):
    headers = {
        "Authorization": f"KakaoAK {api_key}"
    }
    params = {"query": addr}
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    
    response = requests.get(url, headers=headers, params=params)
    time.sleep(0.15)
    if response.status_code == 200:
        documents = response.json().get("documents")
        if documents:
            return float(documents[0]["y"]), float(documents[0]["x"])  # 위도, 경도
    return None, None


def add_to_pos(data):
    tqdm.pandas()

    latitudes = []
    longitudes = []

    for i,(_, row) in enumerate(tqdm(data.iterrows(), total=len(data))):
        if i>0 and i%1000 == 0:
            print(f'{i} rows. 중간 딜레이 5se\n')
    
        
        lat,lng = get_lat_lng(row['소재지지번주소'])
        latitudes.append(lat)
        longitudes.append(lng)

    data['latitude'] = latitudes
    data['longitude'] = longitudes

    #display(data.head(10))
    return data


In [ ]:
parking_area_addr=add_to_pos(parking_area_addr)

In [ ]:
parking_area_addr.dropna(axis=0, inplace=True)

In [ ]:
parking_area_addr.to_csv('공영주차장_위경도.csv')

In [ ]:
parking_area_addr